In [1]:
%pip install PyQt6

Note: you may need to restart the kernel to use updated packages.


In [2]:
%matplotlib qt
import plotly.io as pio
from matplotlib.widgets import Button as MplButton
pio.renderers.default = "notebook"
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Open preprocessed file
# The purpose of this code is to perform trial rejection on the epoched DA069 data
from pathlib import Path
# from bndfpy import open_bndf_as_data_group_file
import bndfpy
from pathlib import Path
from scipy.stats import zscore

# Analysis imports
import numpy as np
import mne
import matplotlib.pyplot as plt
import h5py as h5
import gc as gc

In [3]:
%pip install --extra-index-url https://neuralstorm.github.io/private-pypi/simple/ mlsutilpy


Looking in indexes: https://pypi.org/simple, https://neuralstorm.github.io/private-pypi/simple/
Note: you may need to restart the kernel to use updated packages.


In [4]:
save_dir = Path(r"E:\Projects\iEEG\Reveal\Results\Alec\Figures")
save_dir.mkdir(parents=True, exist_ok=True)

# file_path = Path(r"E:\Projects\iEEG\Reveal\Preprocessed Data\Alec Data\Alec PreProcessedData\DA069_1_preprocessed_Notch_Laplace_epoched.bndf")
file_path = Path("/Users/sreyashravinuthala/Downloads/Alec/DA069_1_preprocessed_epoched.bndf")
# file_path = Path(r"E:\Projects\iEEG\Reveal\Preprocessed Data\DA050_1_preprocessed_epoched.bndf")
# file_path = Path(r"E:\Projects\iEEG\Reveal\Preprocessed Data\DA023_1_preprocessed_epoched.bndf")
# file_path = Path(r"E:\Projects\iEEG\Reveal\Preprocessed Data\DA018_1_preprocessed_epoched.bndf")

bndf = bndfpy.open_bndf_as_data_group_file(file_path)

print("Opened:", file_path)
print("Subject:", bndf.subject_id)
print("Task:", bndf.task_name)
print("Session:", bndf.recording_session)

#Channel rejects (All values are already Zero based for python)
#Rejected trials are done by eye

if "DA069" in str(file_path):
    rejected_channels = [19, 29, 49, 80, 90, 103, 107, 151, 162] # DA069
    seizure_onset_channels = [0, 1, 2, 3, 15, 16, 17, 18, 19, 30, 31, 32, 33, 128, 129, 130, 131, 132, 140, 141, 142, 143, 144] # DA069

elif "DA050" in str(file_path):
    rejected_channels = list(range(62, 128)) + list(range(164, 167)) #DA050
    seizure_onset_channels = [31, 32, 33, 34, 40, 41, 42, 43, 44, 45, 46, 60, 61, 125, 126, 127] #DA050

elif "DA018" in str(file_path):
    rejected_channels = [95, 122, 123, 124, 125] #DA018
    seizure_onset_channels = [0, 1, 2, 3, 4, 5, 6, 20, 21, 22, 33, 34, 35, 49, 50, 51, 52, 102, 103, 104, 105] #DA018

elif "DA023" in str(file_path):
    rejected_channels = [] #DA023
    seizure_onset_channels = [0, 1, 2, 3, 4, 16, 17, 18, 20, 21, 22, 30, 31, 52, 53, 62, 68, 69] #DA023
else:
    raise ValueError("Dataset not found")

Opened: /Users/sreyashravinuthala/Downloads/Alec/DA069_1_preprocessed_epoched.bndf
Subject: DA069
Task: GamblingPlus
Session: 1


In [5]:
# Get epoched Natus data and sample rate directly from the BNDF file
natus_dataset = bndf["neural"]["macro_channels"]["natus"]
natus_sample_freq = natus_dataset.metadata["sample_frequencies"]

print("Dataset shape:", natus_dataset.shape)
print("Natus Sample Freq:", natus_sample_freq)

#Extract Metadata (Sanity Check)
data = np.asarray(natus_dataset[:, :, :])

info = mne.create_info(ch_names=[f"Ch{i + 1}" for i in range(data.shape[1])], sfreq=natus_sample_freq, ch_types="seeg")
epochs = mne.EpochsArray(data, info, tmin=-3)

print("Epoch data shape:", epochs.get_data().shape)
print("Epoch time:", epochs.times[0], "to", epochs.times[-1])

# Get channel labels from Natus metadata
channel_labels = natus_dataset.metadata["location"]

print(channel_labels)
print(channel_labels.shape)

# Real contact labels (W'1, W'2, ...) to go alongside the Ch## index
channel_names = natus_dataset.metadata["labels"]

def get_channel_name(index):
    try:
        name = channel_names[index]
    except (IndexError, TypeError, NameError):
        return epochs.ch_names[index]
    if isinstance(name, bytes):
        name = name.decode(errors="ignore")
    name = str(name).strip()
    if name == "":
        return epochs.ch_names[index]
    return name

print(channel_names[:10])
print("labels:", channel_names.shape, "| location:", channel_labels.shape, "| data channels:", data.shape[1])

Dataset shape: (144, 163, 5377)
Natus Sample Freq: 512.0
Not setting metadata
144 matching events found
No baseline correction applied
0 projection items activated
Epoch data shape: (144, 163, 5377)
Epoch time: -3.0 to 7.5
['R Amygdala' 'R Amygdala' 'R Amygdala' 'R Amygdala' 'R Amygdala'
 'R Amygdala' '' '' '' '' 'R Middle Temporal' 'R Middle Temporal'
 'R Middle Temporal' 'R Middle Temporal' 'OOB' 'R Hippocampus'
 'R Hippocampus' 'R Hippocampus' 'R Hippocampus' 'R Hippocampus'
 'R Hippocampus' '' '' '' '' '' '' '' 'Middle Temporal G' 'OOB'
 'R Hippocampus' 'R Hippocampus' 'R Hippocampus' '' '' '' '' '' '' '' ''
 'R Middle Temporal' '' '' 'R Parahippocampal' 'R Parahippocampal' ''
 'R Inferior Temporal G' '' '' '' '' '' 'R Middle Temporal G'
 'R Middle Temporal G' 'R Middle Temporal G' 'OOB' 'R Temporal Pole' '' ''
 '' '' '' '' 'R Superior Temporal' '' 'R Superior Temporal'
 'R Superior Temporal' 'OOB' 'R Fusiform' 'R Fusiform' 'R Fusiform'
 'R Fusiform' 'R Inferior Tempolal' '' '' '' 

In [6]:
from pathlib import Path

filename = "DA069_1_preprocessed_epoched.bndf"

matches = list(Path.home().rglob(filename))

if matches:
    print("File found!")
    for path in matches:
        print(path)
else:
    print("File NOT found")

File found!
/Users/sreyashravinuthala/Desktop/DA069_1_preprocessed_epoched.bndf
/Users/sreyashravinuthala/Downloads/DA069_1_preprocessed_epoched.bndf
/Users/sreyashravinuthala/Downloads/Alec/DA069_1_preprocessed_epoched.bndf


In [7]:
from pathlib import Path

filename = "DA069_1_preprocessed_epoched.bndf"

matches = list(Path.home().rglob(filename))

if matches:
    print("File found!")
    for path in matches:
        print(path)
else:
    print("File NOT found")

File found!
/Users/sreyashravinuthala/Desktop/DA069_1_preprocessed_epoched.bndf
/Users/sreyashravinuthala/Downloads/DA069_1_preprocessed_epoched.bndf
/Users/sreyashravinuthala/Downloads/Alec/DA069_1_preprocessed_epoched.bndf


In [8]:
# Raw amplitude screen: flag any channel/trial with an absurd jump over 500uV
amp_threshold = 500.0

# #Channel rejects (All values are already Zero based for python)
# #Rejected trials are done by eye
# rejected_channels = [19, 29, 49, 80, 90, 103, 107, 151, 162]  # DA069
# # rejected_channels = list(range(62, 128)) + list(range(164, 167)) #DA050
# #rejected_channels = [] #DA018 Stand in 

# #Add seizure onset zones!!!!!
# seizure_onset_channels = [0, 1, 2, 3, 15, 16, 17, 18, 19, 30, 31, 32, 33, 128, 129, 130, 131, 132, 140, 141, 142, 143, 144] # DA069
# # seizure_onset_channels = [31, 32, 33, 34, 40, 41, 42, 43, 44, 45, 46, 60, 61, 125, 126, 127] #DA050
# #seizure_onset_channels = [0, 1, 2, 3, 4, 5, 6, 20, 21, 22, 33, 34, 35, 49, 50, 51, 52, 102, 103, 104, 105] #DA018




peak = np.abs(epochs.get_data()).max(axis=2)          # (n_trials, n_channels)
artifacts_amp = (peak > amp_threshold).T.astype(float)  # (n_channels, n_trials)

print("Flagged:", int(artifacts_amp.sum()), "/", artifacts_amp.size)

fig, ax = plt.subplots(figsize=(16, 10))
im = ax.imshow(artifacts_amp, aspect="auto", interpolation="nearest", vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label="Amplitude Flag")

for ch in rejected_channels:
    ax.axhspan(ch - 0.5, ch + 0.5, color="red", alpha=0.18)
    ax.hlines(ch, xmin=-0.5, xmax=artifacts_amp.shape[1] - 0.5, color="red", linewidth=1)

for ch in seizure_onset_channels:
    ax.axhspan(ch - 0.5, ch + 0.5, color="blue", alpha=0.18)
    ax.hlines(ch, xmin=-0.5, xmax=artifacts_amp.shape[1] - 0.5, color="red", linewidth=1)

ax.set_xlabel("Trial")
ax.set_ylabel("Channel")
ax.set_title(f"Amplitude Screen (+/- {amp_threshold})")
plt.show()

Flagged: 1242 / 23472


In [9]:
#Defines the bands we are going to use as cell as our freq set up for our mortlet wavelets
qc_bands = {
    "Low": [(0, 30)],
    "Mid": [(30, 56), (64, 115)],
    "High": [(125, 175), (185, 250)]
}

# Mohammad: np.logspace(*np.log10([2, 200]), num=30)
# Adapted only because our High band extends to 250 Hz
freqs = np.logspace(*np.log10([2, 250]), num=30)

# Same as Mohammad
n_cycles = np.floor(np.logspace(*np.log10([3, 10]), num=30))

print("TFR frequencies:", freqs)
print("TFR cycles:", n_cycles)

TFR frequencies: [  2.           2.3623119    2.79025875   3.29573073   3.89277196
   4.59797076   5.43092051   6.41476408   7.57683675   8.94942581
  10.57066754  12.48560685  14.74744882  17.41903691  20.57459908
  24.30181011  28.7042276   33.9041692   40.04611116  47.30070245
  55.86950611  65.99059954  77.94518926  92.06542402 108.74362332
 128.44317765 151.71142345 179.19485041 211.65706367 250.        ]
TFR cycles: [ 3.  3.  3.  3.  3.  3.  3.  4.  4.  4.  4.  4.  4.  5.  5.  5.  5.  6.
  6.  6.  6.  7.  7.  7.  8.  8.  8.  9.  9. 10.]


In [10]:
#Threshold settings
#Mohammad used 7.5 SD, 10 SD is from paper
sd_scale_low = 10
sd_scale_mid = 10
sd_scale_high = 10

# Exact values used by Mohammad
mohammad_min_threshold = 20
mohammad_max_threshold = 45

# Keep False until Mohammad's custom TFR baseline normalization is confirmed
use_mohammad_threshold_clipping = False

if use_mohammad_threshold_clipping:
    min_threshold_low = mohammad_min_threshold
    min_threshold_mid = mohammad_min_threshold
    min_threshold_high = mohammad_min_threshold
    max_threshold_low = mohammad_max_threshold
    max_threshold_mid = mohammad_max_threshold
    max_threshold_high = mohammad_max_threshold
else:
    min_threshold_low = None
    min_threshold_mid = None
    min_threshold_high = None
    max_threshold_low = None
    max_threshold_mid = None
    max_threshold_high = None

analysis_time_range = [-3.5, 3.5]  #This is what the TFR is calculated on
trial_time_range = np.tile(analysis_time_range, (len(epochs), 1))

print("Analysis window:", analysis_time_range)
print("Mohammad 20/45 clipping enabled:", use_mohammad_threshold_clipping)

Analysis window: [-3.5, 3.5]
Mohammad 20/45 clipping enabled: False


In [11]:
#Mohammad's artifact helper functions
def remove_consecutive_numbers(arr,keep_last=False):
    if len(arr) == 0:
        return arr

    diff = np.diff(arr)
    mask_first = np.insert(diff != 1, 0, True)

    if keep_last:
        mask_last = np.append(diff != 1, True)
        mask_last[:-1] = mask_last[:-1] | (diff != 1)
        mask = mask_first | mask_last
    else:
        mask = mask_first

    return arr[mask]


def get_outlier_inds_in_slice(data, time, time_range, threshold,method='clip',mn=None,sd=None, min_threshold=None,keep_last=False, max_threshold=None):
    timeinds = np.argwhere((time >= time_range[0]) & (time <= time_range[1]))
    tmp = data[timeinds].copy()

    if method == 'clip':
        ids = np.argwhere(tmp > threshold)[:,0]
        if ids.size == 0:
            return ids
        else:
            return remove_consecutive_numbers(ids,keep_last=keep_last)

    elif method == 'zscore':
        if mn is None or sd is None:
            mn = np.mean(tmp)
            sd = np.std(tmp)

        thr = mn + threshold*sd

        if min_threshold is not None:
            if thr < min_threshold:
                thr = min_threshold

        if max_threshold is not None:
            if thr > max_threshold:
                thr = max_threshold

        ids = np.argwhere(tmp > thr)[:,0]

        if ids.size == 0:
            return ids
        else:
            return remove_consecutive_numbers(ids,keep_last=keep_last)

#New Method(s) Not finnished 
#So from what i can tell some channels have realy bad data that can have jumps that give liek insnane spikes in power making the means and std like in the millions. This makes the threshold so high that its not flagging channels i know should be excluded
#This may help. Also it could help us confirm channels are bad that we have looked at before by eye
#Also we was thinking we could calculate both mn and _mn and put a magnitude difference. If the difference is to high 
def get_inside_trial_stats(method, data, time, time_range, threshold_reject_mask):
    match method:
        case "all": #all trials is the defualt version that cacluated mn and sd on a trial to trial basis. Not robust to extreme outlires (Should be fine with amplitude rejection)
            Ntrials = data.shape[0]
            
            trial_data = []
            
            for i in range(Ntrials):
                # If threshold reject mask skips 
                if threshold_reject_mask[i] == 1:
                    pass
                else:
                    timeinds = np.argwhere((time >= time_range[i, 0]) & (time <= time_range[i, 1]))
                    trial_data.append(data[i,timeinds].copy())
        
            trial_data = np.concatenate(trial_data)
            mn = np.nanmean(trial_data)
            sd = np.nanstd(trial_data)
    
            return mn, sd
        
        case "quartile":
            Ntrials = data.shape[0]
        
            trial_data = []
            for i in range(Ntrials):
                timeinds = np.argwhere((time >= time_range[i, 0]) & (time <= time_range[i, 1]))
                trial_data.append(data[i,timeinds].copy())
        
            trial_data = np.concatenate(trial_data)
            
            q1 = np.percentile(trial_data, 1)
            q3 = np.percentile(trial_data, 99)
            iqr = q3 - q1
            thr_low = np.max(q1 - 1.5*iqr, 0)
            thr_high = q3 + 1.5*iqr
        
            mask = np.argwhere(trial_data <= thr_high)
            #print(f"Length of valid Inlier samples: {mask.shape[0]} / {trial_data.shape[0]}")
            #print(q1,q3,iqr, thr_low, thr_high)
        
            _mn = np.mean(trial_data[mask])
            _sd = np.std(trial_data[mask]) 
            return _mn, _sd
        
        case "mix":
            Ntrials = data.shape[0]
        
            trial_data = []
            for i in range(Ntrials):
                timeinds = np.argwhere((time >= time_range[i, 0]) & (time <= time_range[i, 1]))
                trial_data.append(data[i,timeinds].copy())
        
            trial_data = np.concatenate(trial_data)
            mn = np.mean(trial_data)
            sd = np.std(trial_data)
        
            q1 = np.percentile(trial_data, 20)
            q3 = np.percentile(trial_data, 80)
            iqr = q3 - q1
            thr_low = np.max(q1 - 1.5*iqr, 0)
            thr_high = q3 + 1.5*iqr
        
            mask = np.argwhere(trial_data <= thr_high)
            print(f"Length of valid samples: {mask.shape[0]} / {trial_data.shape[0]}")
            print(q1,q3,iqr, thr_low, thr_high)
        
            _mn = np.mean(trial_data[mask])
            _sd = np.std(trial_data[mask]) 
        
            print(f"All mean: {mn}, All std: {sd}")
            print(f"Inlier mean: {_mn}, Inlier std: {_sd}")
            # print(np.percentile(trial_data, [10, 25, 75, 90]))
            return mn, sd
        
        case "MAD":
            return mn, sd

In [12]:
# Group each continuous threshold crossing and save its timestamps.
import pandas as pd

artifact_event_columns = ["channel", "trial", "band", "start_sample", "end_sample", "start_s", "end_s", "midpoint_s", "duration_ms", "n_samples", "threshold", "peak_power", "touches_detection_start", "touches_detection_end"]


def get_power_artifact_events(power, times, time_range, threshold, channel, trial, band, sfreq):
    power = np.asarray(power, dtype=float)
    times = np.asarray(times, dtype=float)
    if power.shape != times.shape or not np.isfinite(power).all() or not np.isfinite(threshold):
        raise ValueError(f"Invalid power or threshold: channel {channel + 1}, trial {trial + 1}, {band}.")

    inside = (times >= time_range[0]) & (times <= time_range[1])
    if not inside.any():
        raise ValueError("The detection window contains no samples.")

    # A return to/below threshold always separates two events.
    crossing = (power > threshold) & inside
    edges = np.diff(np.r_[False, crossing, False].astype(np.int8))
    starts = np.flatnonzero(edges == 1)
    stops = np.flatnonzero(edges == -1)
    first, last = np.flatnonzero(inside)[[0, -1]]
    records = []

    for start, stop in zip(starts, stops):
        end = stop - 1
        records.append({
            "channel": int(channel), "trial": int(trial), "band": band,
            "start_sample": int(start), "end_sample": int(end),
            "start_s": float(times[start]), "end_s": float(times[end]),
            "midpoint_s": float((times[start] + times[end]) / 2),
            "duration_ms": float(1000 * (stop - start) / sfreq), "n_samples": int(stop - start),
            "threshold": float(threshold), "peak_power": float(power[start:stop].max()),
            "touches_detection_start": bool(start == first), "touches_detection_end": bool(end == last)
        })

    return records

In [13]:
#Notes
#Maybe have 4 toggle options to see what works best
#1. All Data Average and SD (Not robust to heavy outlire artifacts inflating SD). Uses all samples in all trials 
#2. Only use quartile (Robust to outlires inflating the SD but will always remove some samples in each trial data in its calculation). Uses all non extreme values from all trials 
#3. Use quartile method and all data mean and sd are both calculated and if the magnatidue of using all data is to big we use quartiles over all(more robust to outlires but does get rid of data (depending of quartiles)) 
#4. Use median absolute deviation (similar to what i do for normalizing log space). Uses all samples in all trials but takes the median instead of the mean 

method = "all" # reflects the methods listed above. Options are "all", "quartile", "mix", and "MAD" 

# min_threshold_mid   = 40
# min_threshold_high  = 30
# min_threshold_low   = 20

n_trials = len(epochs)
n_channels = len(epochs.ch_names)

# Event timestamps use the same full detection window and thresholds below.
artifact_detection_settings = None
globals().pop("section_results", None)
artifact_event_work = []

artifact_inds_low = []
artifact_inds_mid = []
artifact_inds_high = []

thr_low = []
thr_mid = []
thr_high = []

low_mask = np.zeros(len(freqs), dtype=bool)
mid_mask = np.zeros(len(freqs), dtype=bool)
high_mask = np.zeros(len(freqs), dtype=bool)

for low, high in qc_bands["Low"]:
    low_mask |= (freqs >= low) & (freqs <= high)

for low, high in qc_bands["Mid"]:
    mid_mask |= (freqs >= low) & (freqs <= high)

for low, high in qc_bands["High"]:
    high_mask |= (freqs >= low) & (freqs <= high)

frq_ids_low = np.argwhere(low_mask).ravel()
frq_ids_mid = np.argwhere(mid_mask).ravel()
frq_ids_high = np.argwhere(high_mask).ravel()

for i in range(n_channels):

    channel_reject_mask = artifacts_amp[i]
    channel_reject_mask = channel_reject_mask.astype(int) # Convert to int

    print(f"Channel {i + 1}/{n_channels}", end="\r")

    channel_epochs = epochs.copy().pick([i])
    tfr_obj = channel_epochs.compute_tfr(method="morlet", freqs=freqs, n_cycles=n_cycles, average=False, return_itc=False)

    low_power = np.squeeze(np.mean(np.squeeze(tfr_obj.data[:,0,frq_ids_low,:]),axis=1))
    mid_power = np.squeeze(np.mean(np.squeeze(tfr_obj.data[:,0,frq_ids_mid,:]),axis=1))
    high_power = np.squeeze(np.mean(np.squeeze(tfr_obj.data[:,0,frq_ids_high,:]),axis=1))

    #channge to calculate different mn and sd methods
    mn_low, sd_low = get_inside_trial_stats(method, low_power, tfr_obj.times, trial_time_range, channel_reject_mask)
    mn_mid, sd_mid = get_inside_trial_stats(method, mid_power, tfr_obj.times, trial_time_range, channel_reject_mask)
    mn_high, sd_high = get_inside_trial_stats(method, high_power, tfr_obj.times, trial_time_range, channel_reject_mask)

    thr_low.append(mn_low + sd_scale_low*sd_low)
    thr_mid.append(mn_mid + sd_scale_mid*sd_mid)
    thr_high.append(mn_high + sd_scale_high*sd_high)

    artifact_inds_low.append([])
    artifact_inds_mid.append([])
    artifact_inds_high.append([])

    for i_trial in range(n_trials):

        tmp_tfr = np.squeeze(tfr_obj.data[i_trial,0,:,:].copy())

        artifact_inds_low[i].append(get_outlier_inds_in_slice(np.squeeze(np.mean(tmp_tfr[frq_ids_low,:],axis=0)),
                                                              tfr_obj.times, analysis_time_range,
                                                              threshold=sd_scale_low, method='zscore',mn=mn_low,sd=sd_low,
                                                              min_threshold=min_threshold_low,max_threshold=max_threshold_low))

        artifact_inds_mid[i].append(get_outlier_inds_in_slice(np.squeeze(np.mean(tmp_tfr[frq_ids_mid,:],axis=0)),
                                                              tfr_obj.times, analysis_time_range,
                                                              threshold=sd_scale_mid, method='zscore',mn=mn_mid,sd=sd_mid,
                                                              min_threshold=min_threshold_mid,max_threshold=max_threshold_mid))

        artifact_inds_high[i].append(get_outlier_inds_in_slice(np.squeeze(np.mean(tmp_tfr[frq_ids_high,:],axis=0)),
                                                               tfr_obj.times, analysis_time_range,
                                                               threshold=sd_scale_high, method='zscore',mn=mn_high,sd=sd_high,
                                                               min_threshold=min_threshold_high,max_threshold=max_threshold_high))


        # Record each separate crossing in each band, including single samples.
        for event_band, event_ids, event_mean, event_sd, event_k, event_min, event_max in [
            ("Low", frq_ids_low, mn_low, sd_low, sd_scale_low, min_threshold_low, max_threshold_low),
            ("Mid", frq_ids_mid, mn_mid, sd_mid, sd_scale_mid, min_threshold_mid, max_threshold_mid),
            ("High", frq_ids_high, mn_high, sd_high, sd_scale_high, min_threshold_high, max_threshold_high)
        ]:
            event_threshold = event_mean + event_k * event_sd
            if event_min is not None:
                event_threshold = max(event_threshold, event_min)
            if event_max is not None:
                event_threshold = min(event_threshold, event_max)
            event_power = np.mean(tmp_tfr[event_ids, :], axis=0)
            artifact_event_work.extend(get_power_artifact_events(event_power, tfr_obj.times, analysis_time_range, event_threshold, i, i_trial, event_band, float(epochs.info["sfreq"])))

# Store completed results; changing the section later does not refit thresholds.
artifact_events = pd.DataFrame(artifact_event_work, columns=artifact_event_columns)
artifact_detection_settings = {
    "analysis_time_range": list(analysis_time_range), "trial_time_range": np.asarray(trial_time_range).copy(),
    "n_channels": n_channels, "n_trials": n_trials, "sfreq": float(epochs.info["sfreq"]),
    "times": epochs.times.copy(), "method": method, "k": [sd_scale_low, sd_scale_mid, sd_scale_high],
    "freqs": np.asarray(freqs).copy(), "n_cycles": np.asarray(n_cycles).copy()
}
print(f"\nSaved {len(artifact_events):,} band events in artifact_events.")

print("\nDone")

Channel 163/163
Saved 17,587 band events in artifact_events.

Done


In [14]:
print(min(thr_low))
print(min(thr_mid))
print(min(thr_high))


4.556337286448642
4.729799462832376
3.63069495767198


In [15]:
for event_band, event_ids, event_mean, event_sd, event_k, event_min, event_max in [
            ("Low", frq_ids_low, mn_low, sd_low, sd_scale_low, min_threshold_low, max_threshold_low),
            ("Mid", frq_ids_mid, mn_mid, sd_mid, sd_scale_mid, min_threshold_mid, max_threshold_mid),
            ("High", frq_ids_high, mn_high, sd_high, sd_scale_high, min_threshold_high, max_threshold_high)
        ]:
    # event_threshold = event_mean + event_k * event_sd
    print(event_mean, event_k, event_sd)

19402.45537358816 10 43734.94677979791
153.96347511963677 10 1628.4947587004767
5.3601315812918005 10 19.68593785934697


In [16]:
artifacts_low = np.zeros((len(artifact_inds_low),len(artifact_inds_low[0])))
for i in range(len(artifact_inds_low)):
    for j in range(len(artifact_inds_low[0])):
        if artifact_inds_low[i][j].size > 0:
            artifacts_low[i][j] = 1


artifacts_mid = np.zeros((len(artifact_inds_mid),len(artifact_inds_mid[0])))
for i in range(len(artifact_inds_mid)):
    for j in range(len(artifact_inds_mid[0])):
        if artifact_inds_mid[i][j].size > 0:
            artifacts_mid[i][j] = 1


artifacts_high = np.zeros((len(artifact_inds_high),len(artifact_inds_high[0])))
for i in range(len(artifact_inds_high)):
    for j in range(len(artifact_inds_high[0])):
        if artifact_inds_high[i][j].size > 0:
            artifacts_high[i][j] = 1


artifacts = artifacts_low + artifacts_mid + artifacts_high

print("Artifact matrix shape:", artifacts.shape)
print("Flagged channel/trial pairs:", np.sum(artifacts > 0))

Artifact matrix shape: (163, 144)
Flagged channel/trial pairs: 7996


In [17]:
print(artifacts_low.sum())
print(artifacts_mid.sum())
print(artifacts_high.sum())

1750.0
3614.0
6481.0


In [18]:
plt.figure(figsize=(16, 10))

plt.imshow(artifacts, aspect="auto", interpolation="nearest", vmin=0, vmax=3)

plt.colorbar(label="Number of QC Bands Flagged")
plt.xlabel("Trial")
plt.ylabel("Channel")
plt.title("Artifact Matrix - Channel x Trial")
plt.savefig(save_dir / f"{bndf.subject_id}_ArtifactMatrix_AllBands.png", dpi=300, bbox_inches="tight")

plt.show()

In [19]:
# Plot artifact matrix and highlight manually rejected DA069 channels

fig, ax = plt.subplots(figsize=(16, 10))

im = ax.imshow(artifacts, aspect="auto", interpolation="nearest", vmin=0, vmax=3)

plt.colorbar(im, ax=ax, label="Number of QC Bands Flagged")

for ch in rejected_channels:
    ax.axhspan(ch - 0.5, ch + 0.5, color="red", alpha=0.18)
    ax.hlines(ch, xmin=-0.5, xmax=artifacts.shape[1] - 0.5, color="red", linewidth=1)

for ch in seizure_onset_channels:
    ax.axhspan(ch - 0.5, ch + 0.5, color="blue", alpha=0.18)
    ax.hlines(ch, xmin=-0.5, xmax=artifacts_amp.shape[1] - 0.5, color="red", linewidth=1)
    
ax.set_xlabel("Trial")
ax.set_ylabel("Channel")
ax.set_title("Artifact Matrix - Channel x Trial (Rejected Channels Highlighted - RED = Rejected Channels, BLUE = SoZ)")
fig.savefig(save_dir / f"{bndf.subject_id}_ArtifactMatrix_RejectedChannelsHighlighted.png", dpi=300, bbox_inches="tight")

plt.show()

In [20]:
print(save_dir)

E:\Projects\iEEG\Reveal\Results\Alec\Figures


In [21]:
# # Plot artifact matrix excluding rejected channels and seizure onset zone channels

# excluded_channels = sorted(set(rejected_channels + seizure_onset_channels))
# kept_channels = [ch for ch in range(artifacts.shape[0]) if ch not in excluded_channels]

# artifacts_clean = artifacts[kept_channels, :]

# fig, ax = plt.subplots(figsize=(16, 10))

# im = ax.imshow(artifacts_clean, aspect="auto", interpolation="nearest", vmin=0, vmax=3)

# plt.colorbar(im, ax=ax, label="Number of QC Bands Flagged")

# ax.set_xlabel("Trial")
# ax.set_ylabel("Channel")
# ax.set_title("Artifact Matrix - Rejected and SOZ Channels Excluded")

# # Show original channel numbers on the y-axis
# tick_step = 5
# tick_positions = np.arange(0, len(kept_channels), tick_step)
# ax.set_yticks(tick_positions)
# ax.set_yticklabels([kept_channels[i] for i in tick_positions])

# fig.savefig(save_dir / f"{bndf.subject_id}_ArtifactMatrix_ExcludedChannels.png", dpi=300, bbox_inches="tight")

# plt.show()

In [22]:
# Plot Low, Mid, and High artifact matrices separately

band_matrices = [
    ("Low", artifacts_low),
    ("Mid", artifacts_mid),
    ("High", artifacts_high)
]

for band_name, band_matrix in band_matrices:

    fig, ax = plt.subplots(figsize=(16, 10))

    im = ax.imshow(band_matrix, aspect="auto", interpolation="nearest", vmin=0, vmax=1)

    plt.colorbar(im, ax=ax, label="Artifact Flag")

    for ch in rejected_channels:
        ax.axhspan(ch - 0.5, ch + 0.5, color="red", alpha=0.18)
        ax.hlines(ch, xmin=-0.5, xmax=band_matrix.shape[1] - 0.5, color="red", linewidth=1)

    for ch in seizure_onset_channels:
        ax.axhspan(ch - 0.5, ch + 0.5, color="blue", alpha=0.18)
        ax.hlines(ch, xmin=-0.5, xmax=artifacts_amp.shape[1] - 0.5, color="red", linewidth=1)

    ax.set_xlabel("Trial")
    ax.set_ylabel("Channel")
    ax.set_title(f"{band_name} Band Artifact Matrix - Channel x Trial")

    fig.savefig(save_dir / f"{bndf.subject_id}_ArtifactMatrix_{band_name}Band.png", dpi=300, bbox_inches="tight")

    plt.show()

In [23]:
# Change these three settings, then rerun the next calculation and plot cells.
section_time_range = [0, 1.0]
#artifact_buffer_ms = {"Low": 1000, "Mid": 1000, "High": 1000}  # Distance from either section edge, in ms. Conservative estimate based on lowest wavelet and cycles
artifact_buffer_ms = {"Low": 1000, "Mid": 150, "High": 50}  # Distance from either section edge, in ms Based on half widths of the lowest freq in a band with wavelets from how MNE calculates wavelets
# section_included_bands = ["Low", "Mid", "High"]  # e.g. ["Mid", "High"]
section_included_bands = ["Low", "Mid"]  # e.g. ["Mid", "High"]

In [24]:
def make_section_artifact_results(events, detection_settings, section_time_range, buffer_ms, included_bands, rejected, soz):
    band_names = ["Low", "Mid", "High"]
    section = np.asarray(section_time_range, dtype=float)
    detection = np.asarray(detection_settings["analysis_time_range"], dtype=float)
    n_ch, n_tr = detection_settings["n_channels"], detection_settings["n_trials"]

    if section.shape != (2,) or not np.isfinite(section).all() or section[0] >= section[1]:
        raise ValueError("section_time_range must contain two increasing finite times.")
    selected = list(included_bands)
    if not selected or len(set(selected)) != len(selected) or any(band not in band_names for band in selected):
        raise ValueError('included_bands must contain distinct names from "Low", "Mid", "High", with at least one selected.')
    if set(buffer_ms) != set(band_names) or any(not np.isfinite(buffer_ms[b]) or buffer_ms[b] < 0 for b in band_names):
        raise ValueError("buffer_ms must specify a finite nonnegative value for Low, Mid and High.")

    # All three band plots need complete detection coverage, including buffers.
    largest_buffer = max(buffer_ms.values()) / 1000
    if section[0] - largest_buffer < detection[0] - 1e-12 or section[1] + largest_buffer > detection[1] + 1e-12:
        raise ValueError("The selected window plus buffer extends outside the completed detection window. Reduce the selection/buffer, or widen analysis_time_range and rerun detection first.")

    excluded = sorted(set(rejected) | set(soz))
    if any(not isinstance(ch, (int, np.integer)) or ch < 0 or ch >= n_ch for ch in excluded):
        raise ValueError("Excluded channel indices must be zero-based integers within the recording.")
    kept = np.array([ch for ch in range(n_ch) if ch not in excluded], dtype=int)

    table = events.copy()
    for column in ["channel", "trial", "n_samples"]:
        table[column] = table[column].astype(int)
    for column in ["start_s", "end_s", "midpoint_s", "duration_ms"]:
        table[column] = table[column].astype(float)

    # Use the complete detected interval for overlap/proximity, not only its midpoint.
    table["distance_to_section_ms"] = 1000 * np.maximum(0, np.maximum(section[0] - table["end_s"], table["start_s"] - section[1]))
    table["in_section"] = (table["end_s"] >= section[0]) & (table["start_s"] <= section[1])
    table["within_buffer"] = (table["distance_to_section_ms"] <= table["band"].map(buffer_ms) + 1e-9) & ~table["in_section"]
    table["counts_for_section"] = table["in_section"] | table["within_buffer"]
    table["channel_included"] = ~table["channel"].isin(excluded)
    table["selected_band"] = table["band"].isin(selected)
    table["used_in_combined"] = table["counts_for_section"] & table["channel_included"] & table["selected_band"]

    matrices = {}
    for band in band_names:
        matrix = np.zeros((n_ch, n_tr), dtype=bool)
        matching = table.loc[(table["band"] == band) & table["counts_for_section"]]
        matrix[matching["channel"].to_numpy(dtype=int), matching["trial"].to_numpy(dtype=int)] = True
        # NaN keeps exclusions distinct from evaluated, unflagged pairs.
        masked = matrix.astype(float)
        masked[excluded, :] = np.nan
        matrices[band] = masked

    combined = np.sum(np.stack([matrices[band] for band in selected]), axis=0)
    flag = np.where(np.isfinite(combined), (combined > 0).astype(float), np.nan)
    return {
        "events": table, "band_matrices_full": matrices,
        "band_matrices": {band: matrix[kept] for band, matrix in matrices.items()},
        "combined_full": combined, "combined": combined[kept],
        "flags_full": flag, "flags": flag[kept], "channel_indices": kept,
        "settings": {"section_time_range": section.copy(), "buffer_ms": dict(buffer_ms), "included_bands": selected, "excluded_channels": excluded}
    }


# Classify timestamped events against the section without recomputing power.
if "artifact_events" not in globals() or globals().get("artifact_detection_settings") is None:
    raise RuntimeError("Run the detection cell to calculate event timestamps first.")
if (len(epochs.ch_names), len(epochs)) != (artifact_detection_settings["n_channels"], artifact_detection_settings["n_trials"]) or not np.array_equal(epochs.times, artifact_detection_settings["times"]):
    raise RuntimeError("Epochs have changed. Rerun detection first.")
if not np.array_equal(analysis_time_range, artifact_detection_settings["analysis_time_range"]):
    raise RuntimeError("The detection window has changed. Rerun detection first.")

section_results = make_section_artifact_results(artifact_events, artifact_detection_settings, section_time_range, artifact_buffer_ms, section_included_bands, rejected_channels, seizure_onset_channels)
section_artifact_events = section_results["events"].loc[section_results["events"]["used_in_combined"]].copy()
section_artifacts_low = section_results["band_matrices"]["Low"]
section_artifacts_mid = section_results["band_matrices"]["Mid"]
section_artifacts_high = section_results["band_matrices"]["High"]
section_artifacts = section_results["combined"]
section_artifact_flags = section_results["flags"]
section_channel_indices = section_results["channel_indices"]
section_applied_settings = section_results["settings"]

print(f"Detection window: {artifact_detection_settings['analysis_time_range']} s")
print(f"Selected window: {section_applied_settings['section_time_range']} s | Buffer (ms): {section_applied_settings['buffer_ms']}")
print(f"Included bands: {section_applied_settings['included_bands']}")
print(f"Channels retained: {len(section_channel_indices)} | Flagged pairs: {int(np.sum(section_artifact_flags))}/{section_artifact_flags.size}")

Detection window: [-3.5, 3.5] s
Selected window: [0. 1.] s | Buffer (ms): {'Low': 1000, 'Mid': 150, 'High': 50}
Included bands: ['Low', 'Mid']
Channels retained: 132 | Flagged pairs: 648/19008


In [25]:
# Combined selected-window matrix, followed by all three band matrices.
def plot_section_artifact_matrix(values, title, vmax, label):
    if not len(section_channel_indices):
        print("All channels were excluded; no selected-window matrix to plot.")
        return
    fig, ax = plt.subplots(figsize=(16, 10))
    im = ax.imshow(values, aspect="auto", interpolation="nearest", cmap="viridis", vmin=0, vmax=vmax)
    fig.colorbar(im, ax=ax, label=label, ticks=np.arange(vmax + 1))
    tick_positions = np.arange(0, len(section_channel_indices), 5)
    ax.set_yticks(tick_positions)
    ax.set_yticklabels(section_channel_indices[tick_positions])
    ax.set_xlabel("Trial (zero-based)")
    ax.set_ylabel("Original channel (zero-based; excluded channels omitted)")
    ax.set_title(title)
    plt.show()
    plt.close(fig)


section_title = f"{section_applied_settings['section_time_range'][0]:g} to {section_applied_settings['section_time_range'][1]:g} s"
plot_section_artifact_matrix(section_artifacts, f"Selected-window Artifact Matrix | {section_title}\nBands: {', '.join(section_applied_settings['included_bands'])}", len(section_applied_settings["included_bands"]), "Number of selected QC bands flagged")

# Always show all three bands, regardless of which enter the combined matrix.
for section_band in ["Low", "Mid", "High"]:
    plot_section_artifact_matrix(section_results["band_matrices"][section_band], f"{section_band} Band | {section_title} | Buffer: {section_applied_settings['buffer_ms'][section_band]:g} ms", 1, "Artifact flag")

In [26]:
# !pip install ipywidgets

In [39]:
# Browse TFR trials for one channel using Previous/Next buttons
# Note: Turns out putting it into to dB log space and then zscoring that doesn't make a lot a sense. But i heard there is still a way to normalize in log space. 
# Also to the widget i want to add anataomy label. if there is no label it is WM if not specified. Also i want to show sd and mean for all or quartiled mean and sd depeding of method used
from scipy.stats import zscore
from ipywidgets import Button, HBox, Output
from IPython.display import display, clear_output
 
trial_index=0
channel_index=0  # Zero-based: 0 = Ch1
view_decim=1
normalized_db = True #Choices: False = linear zscored representation without log dB space(May remove), True = normalized dB log space (Better Option)
#Note: Linear Zscored representation is not very usefull. Mean and SD come from the trial itself, so a big artifact inflates its own SD and gets scaled back down to looking normal. Trial by trials are less comparable also with color map range
#note: normalized log scale is centered on the median and scaled by the spread across all trials, so a bad trial stands out against the rest of the channel instead of hiding itself
colormap_range = 5 

view_times=epochs.times[::view_decim]

frequency_ticks=np.array([2,4,10,30,70,100,200,250])
frequency_ticks=frequency_ticks[(frequency_ticks>=freqs[0]) & (frequency_ticks<=freqs[-1])]

# Anatomy label from Natus metadata; blank/missing entries fall back to WM
def get_anatomy_label(index):
    try:
        label = channel_labels[index]
    except (IndexError, TypeError, NameError):
        return "WM"
    if isinstance(label, bytes):
        label = label.decode(errors="ignore")
    label = str(label).strip()
    if label == "" or label.lower() in {"nan", "none", "n/a", "na", "unknown"}:
        return "WM"
    return label

# Wrapped in a function so the channel buttons can re-run it. Same lines as before, they
# just needed to happen again when channel_index changes instead of only once.
def load_channel():
    global channel_epochs, channel_tfr_view, channel_tfr_db, baseline_per_freq, spread_per_freq, raw_limit, anatomy_label

    channel_epochs=epochs.get_data(picks=[channel_index])
    channel_tfr_view=mne.time_frequency.tfr_array_morlet(channel_epochs, sfreq=epochs.info["sfreq"], freqs=freqs, n_cycles=n_cycles, output="power", use_fft=True, decim=view_decim, n_jobs=1, verbose=False)

    anatomy_label = get_anatomy_label(channel_index)

    # Per-frequency center and spread in log space, pooled across all trials and time.
    # Pooling across trials (not within one trial) is what lets a bad trial actually look bad
    # against the rest of the channel. Median for the center so a few huge jumps don't drag it.
    channel_tfr_db = 10*np.log10(channel_tfr_view[:, 0, :, :] + np.finfo(float).eps)
    baseline_per_freq = np.median(channel_tfr_db, axis=(0, 2))
    spread_per_freq = np.std(channel_tfr_db, axis=(0, 2))
    # Robust alternative if the SD is still getting inflated by spikes:
    # spread_per_freq = 1.4826*np.median(np.abs(channel_tfr_db - baseline_per_freq[None,:,None]), axis=(0, 2))

    # Fixed y-limits for the raw trace so the amplitude scale doesn't rescale per trial
    raw_limit = np.percentile(np.abs(channel_epochs), 99.9)

previous_button=Button(description="Previous trial")
next_button=Button(description="Next trial")
previous_channel_button=Button(description="Previous channel")
next_channel_button=Button(description="Next channel")
plot_output=Output()
state={"trial_index":trial_index}

def add_artifact_event_overlay(axes, channel, trial):
    if "artifact_events" not in globals() or globals().get("artifact_detection_settings") is None:
        print("Run detection to show event timestamps.")
        return

    events = artifact_events.loc[(artifact_events["channel"] == channel) & (artifact_events["trial"] == trial)]
    colors = {"Low": "darkorange", "Mid": "magenta", "High": "deepskyblue"}

    if "section_results" in globals():
        applied = section_results["settings"]
        start, end = applied["section_time_range"]

        buffer_alpha = {"Low": 0.10, "Mid": 0.18, "High": 0.28}

        for ax in axes:
            for band in applied["included_bands"]:
                buffer = applied["buffer_ms"][band] / 1000
                if buffer > 0:
                    ax.axvspan(
                        start - buffer, start,
                        color="gray", alpha=buffer_alpha[band],
                        label=f"{band} buffer ({buffer * 1000:g} ms)"
                        if ax is axes[0] else None,
                    )
                    ax.axvspan(
                        end, end + buffer,
                        color="gray", alpha=buffer_alpha[band],
                    )

            ax.axvspan(start, end, color="green", alpha=0.08)
            ax.axvline(start, color="green", linestyle="-.")
            ax.axvline(
                end, color="green", linestyle="-.",
                label="Selected window" if ax is axes[0] else None,
            )

            
        if channel in applied["excluded_channels"]:
            print("Channel excluded from section matrices (by-eye/SOZ).")
        else:
            print(f"Section: {start:g} to {end:g} s | Bands flagged: {int(section_results['combined_full'][channel, trial])} | Included: {applied['included_bands']}")

    for ax in axes:
        for band, color in colors.items():
            for number, event in enumerate(events.loc[events["band"] == band].itertuples()):
                ax.axvspan(event.start_s, event.end_s, color=color, alpha=0.12)
                ax.axvline(event.midpoint_s, color=color, linestyle="--", linewidth=0.8, label=f"{band} midpoint" if number == 0 and ax is axes[0] else None)

    if len(events):
        print("Spans = threshold crossings; dashed lines = midpoints. Times are epoch-relative seconds.")
        display(events[["band", "start_s", "end_s", "midpoint_s", "duration_ms"]].round(4))
    else:
        print("No threshold crossings in this channel/trial.")

plot_window = {"fig": None}
# Store review decisions for each channel/trial
import csv
from pathlib import Path

review_file = Path("DA069_1_review_decisions.csv")

review_decisions = {}

if review_file.exists():
    with review_file.open(newline="") as f:
        for row in csv.DictReader(f):
            key = (int(row["channel_index"]), int(row["trial_index"]))
            review_decisions[key] = row["decision"]

def save_reviews():
    with review_file.open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["channel_index", "trial_index", "decision"])

        for (channel, trial), decision in sorted(review_decisions.items()):
            writer.writerow([channel, trial, decision])

# Qt dialog for editing the section-artifact settings from the plot window.
def open_artifact_settings():
    from PyQt6.QtWidgets import (QApplication, QDialog, QVBoxLayout, QGridLayout,
                                 QLabel, QCheckBox, QSpinBox, QDialogButtonBox,
                                 QMessageBox)

    app = QApplication.instance()
    if app is None:
        raise RuntimeError("Use the Qt Matplotlib backend (%matplotlib qt) first.")

    dialog = QDialog()
    dialog.setWindowTitle("Artifact Detection Settings")
    layout = QVBoxLayout(dialog)
    layout.addWidget(QLabel("Select bands and buffer lengths for the selected-window artifact results."))
    grid = QGridLayout()
    grid.addWidget(QLabel("Include"), 0, 0)
    grid.addWidget(QLabel("Band"), 0, 1)
    grid.addWidget(QLabel("Buffer (ms)"), 0, 2)
    enabled, buffer_inputs = {}, {}
    for row, band in enumerate(("Low", "Mid", "High"), start=1):
        check = QCheckBox()
        check.setChecked(band in section_included_bands)
        spin = QSpinBox()
        spin.setRange(0, 100000)
        spin.setValue(int(artifact_buffer_ms[band]))
        enabled[band], buffer_inputs[band] = check, spin
        grid.addWidget(check, row, 0)
        grid.addWidget(QLabel(band), row, 1)
        grid.addWidget(spin, row, 2)
    layout.addLayout(grid)
    layout.addWidget(QLabel("Changes apply to section classification, not threshold detection."))
    buttons = QDialogButtonBox(QDialogButtonBox.StandardButton.Apply |
                               QDialogButtonBox.StandardButton.Close)
    layout.addWidget(buttons)
    buttons.rejected.connect(dialog.reject)

    def apply_changes():
        global artifact_buffer_ms, section_included_bands, section_results
        global section_artifact_events, section_artifacts_low, section_artifacts_mid
        global section_artifacts_high, section_artifacts, section_artifact_flags
        global section_channel_indices, section_applied_settings

        selected = [b for b in ("Low", "Mid", "High") if enabled[b].isChecked()]
        buffers = {b: buffer_inputs[b].value() for b in ("Low", "Mid", "High")}
        if not selected:
            QMessageBox.warning(dialog, "Invalid selection", "Select at least one frequency band.")
            return
        try:
            updated = make_section_artifact_results(
                artifact_events, artifact_detection_settings, section_time_range,
                buffers, selected, rejected_channels, seizure_onset_channels
            )
        except (ValueError, KeyError, RuntimeError) as exc:
            QMessageBox.warning(dialog, "Cannot apply settings", str(exc))
            return

        # Commit only after validation and successful recomputation.
        artifact_buffer_ms = buffers
        section_included_bands = selected
        section_results = updated
        section_artifact_events = updated["events"].loc[updated["events"]["used_in_combined"]].copy()
        section_artifacts_low = updated["band_matrices"]["Low"]
        section_artifacts_mid = updated["band_matrices"]["Mid"]
        section_artifacts_high = updated["band_matrices"]["High"]
        section_artifacts = updated["combined"]
        section_artifact_flags = updated["flags"]
        section_channel_indices = updated["channel_indices"]
        section_applied_settings = updated["settings"]
        dialog.accept()
        show_trial()

    buttons.button(QDialogButtonBox.StandardButton.Apply).clicked.connect(apply_changes)
    dialog.exec()

def show_trial():
    trial_index=state["trial_index"]
    #trial_power_db=10*np.log10(channel_tfr_view[trial_index,0,:,:]+np.finfo(float).eps)

    if normalized_db:
        # Log-space z-score equivilent: convert to dB first, then center and scale in the log domain.
        # Subtracting the per-frequency median removes 1/f, dividing by the per-frequency
        # spread makes every frequency dimensionless and directly comparable.
        plot_tfr=10*np.log10(channel_tfr_view[trial_index,0,:,:]+np.finfo(float).eps) #added epsilon so the log of 0 doesn't go to inf
        plot_tfr=(plot_tfr-baseline_per_freq[:,None])/spread_per_freq[:,None]
        plot_tfr=np.nan_to_num(plot_tfr)
        colorbar_label="Normalized Log power (SD from channel median)"
    else:
        # Z-score each frequency across time
        plot_tfr=zscore(channel_tfr_view[trial_index,0,:,:],axis=1,nan_policy="omit")
        plot_tfr=np.nan_to_num(plot_tfr)
        colorbar_label="Normalized Linear Power (a.u.)"

    with plot_output:
        clear_output(wait=True)

    
        if plot_window["fig"] is None or not plt.fignum_exists(plot_window["fig"].number):
            fig = plt.figure(figsize=(14, 9))
            plot_window["fig"] = fig
        else:
            fig = plot_window["fig"]
            fig.clear()

        ax, ax_raw = fig.subplots(
            2, 1,
            sharex=True,
            gridspec_kw={"height_ratios": [3, 1]}
        )
        fig.subplots_adjust(bottom=0.20)

        im=ax.pcolormesh(view_times,freqs,plot_tfr,shading="auto", vmin=-colormap_range, vmax=colormap_range)

    

        ax.axvline(0,color="black",linestyle="--",label="Choice")
        ax.axvline(analysis_time_range[0],color="red",linestyle=":")
        ax.axvline(analysis_time_range[1],color="red",linestyle=":",label="Threshold window")

        ax.set_xlim(view_times[0],view_times[-1])
        ax.set_yscale("log")
        def format_spectrogram_coords(x, y):
            if not (np.isfinite(x) and np.isfinite(y)):
                return ""

            # Find the nearest time and frequency
            t_idx = np.abs(view_times - x).argmin()
            f_idx = np.abs(freqs - y).argmin()

            # Get the actual power value from the spectrogram
            power = plot_tfr[f_idx, t_idx]

            return (
                f"Time: {x:.3f} s | "
                f"Frequency: {y:.1f} Hz | "
                f"Power: {power:.3f}"
            )

        ax.format_coord = format_spectrogram_coords


        ax.set_ylim(freqs[0],freqs[-1])
        ax.set_yticks(frequency_ticks)
        ax.set_yticklabels([f"{frequency:g}" for frequency in frequency_ticks])

        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Frequency (Hz)")
        ax.set_title(f"Trial {trial_index+1} of {len(epochs)} — {epochs.ch_names[channel_index]} — {get_channel_name(channel_index)} ({anatomy_label})")
        decision = review_decisions.get((channel_index, trial_index))

        if decision == "Accepted":
            ax.set_title(ax.get_title() + " | ✓ ACCEPTED", color="green")
        elif decision == "Rejected":
            ax.set_title(ax.get_title() + " | ✗ REJECTED", color="red")
        else:
            ax.set_title(ax.get_title() + " | NOT REVIEWED")
        

        # Raw time series for the same trial, sharing the time axis with the TFR above
        ax_raw.plot(epochs.times,channel_epochs[trial_index,0,:],color="black",linewidth=0.5)
        ax_raw.axvline(0,color="black",linestyle="--")
        ax_raw.axvline(analysis_time_range[0],color="red",linestyle=":")
        ax_raw.axvline(analysis_time_range[1],color="red",linestyle=":")
        #ax_raw.set_ylim(-raw_limit,raw_limit)
        ax_raw.set_xlabel("Time (s)")
        ax_raw.set_ylabel("Amplitude")

        add_artifact_event_overlay((ax, ax_raw), channel_index, trial_index)
        
        ax.legend(
            loc="upper left",
            bbox_to_anchor=(1.19, 1),
            fontsize=8
        )

        fig.colorbar(im,ax=[ax,ax_raw],label=colorbar_label)
        # Create navigation buttons in the external window
        # Trial navigation buttons
       # Trial navigation buttons — TOP row (larger, more prominent)
        # Trial navigation — bottom center
        ax_prev = fig.add_axes([0.30, 0.075, 0.18, 0.045])
        ax_next = fig.add_axes([0.52, 0.075, 0.18, 0.045])

        # Channel navigation — small buttons, top right
        ax_prev_ch = fig.add_axes([0.73, 0.945, 0.11, 0.032])
        ax_next_ch = fig.add_axes([0.85, 0.945, 0.11, 0.032])
        ax_settings = fig.add_axes([0.60, 0.945, 0.11, 0.032])



        btn_prev = MplButton(ax_prev, "Previous Trial",
                     color="#D0D5DD", hovercolor="#B8C0CC")
        btn_next = MplButton(ax_next, "Next Trial",
                            color="#D0D5DD", hovercolor="#B8C0CC")

        btn_prev_ch = MplButton(ax_prev_ch, "Previous Channel",
                                color="#ECEFF3", hovercolor="#D0D5DD")
        btn_next_ch = MplButton(ax_next_ch, "Next Channel",
                                color="#ECEFF3", hovercolor="#D0D5DD")

        btn_settings = MplButton(ax_settings, "Settings",
                                 color="#ECEFF3", hovercolor="#D0D5DD")
        btn_settings.label.set_fontsize(8)

        # Accept / Reject buttons — bottom row
        ax_accept = fig.add_axes([0.30, 0.015, 0.18, 0.045])
        ax_reject = fig.add_axes([0.52, 0.015, 0.18, 0.045])

        btn_accept = MplButton(ax_accept, "✓ Accept Trial", color="#B8E6C3", hovercolor="#8BD39C")
        btn_reject = MplButton(ax_reject, "✗ Reject Trial", color="#F3B5B5", hovercolor="#E98C8C")

                # Small gray Clear Decision button — bottom right
        ax_clear = fig.add_axes([0.78, 0.020, 0.14, 0.035])

        btn_clear = MplButton(
            ax_clear,
            "Clear Decision",
            color="#ECEFF3",
            hovercolor="#D0D5DD"
        )
        btn_clear.label.set_fontsize(8)

        btn_prev_ch.label.set_fontsize(8)
        btn_next_ch.label.set_fontsize(8)

        # Keep buttons alive when the graph updatesplot_window["buttons"] = (btn_prev, btn_next)
        plot_window["buttons"] = (
            btn_prev, btn_next,
            btn_prev_ch, btn_next_ch,
            btn_accept, btn_reject, btn_clear, btn_settings
        )

        def go_previous(event):
            state["trial_index"] = max(0, state["trial_index"] - 1)
            show_trial()

        def go_next(event):
            state["trial_index"] = min(len(epochs) - 1, state["trial_index"] + 1)
            show_trial()

        btn_prev.on_clicked(go_previous)
        btn_next.on_clicked(go_next)

        def accept_trial(event):
            review_decisions[(channel_index, state["trial_index"])] = "Accepted"
            save_reviews()
            show_trial()

        def reject_trial(event):
            review_decisions[(channel_index, state["trial_index"])] = "Rejected"
            save_reviews()
            show_trial()

        btn_accept.on_clicked(accept_trial)
        btn_reject.on_clicked(reject_trial)
        def clear_decision(event):
            key = (channel_index, state["trial_index"])
            review_decisions.pop(key, None)
            save_reviews()
            show_trial()

        btn_clear.on_clicked(clear_decision)

        def go_previous_channel(event):
            previous_channel(None)
        def go_next_channel(event):
            next_channel(None)

        btn_prev_ch.on_clicked(go_previous_channel)
        btn_next_ch.on_clicked(go_next_channel)
        btn_settings.on_clicked(lambda event: open_artifact_settings())

        fig.canvas.draw_idle()
        plt.show(block=False)

        previous_button.disabled=trial_index==0
        next_button.disabled=trial_index==len(epochs)-1

def previous_trial(_):
    state["trial_index"]=max(0,state["trial_index"]-1)
    show_trial()

def next_trial(_):
    state["trial_index"]=min(len(epochs)-1,state["trial_index"]+1)
    show_trial()

def previous_channel(_):
    global channel_index
    channel_index=max(0,channel_index-1)
    load_channel()
    show_trial()

def next_channel(_):
    global channel_index
    channel_index=min(len(epochs.ch_names)-1,channel_index+1)
    load_channel()
    show_trial()

previous_button.on_click(previous_trial)
next_button.on_click(next_trial)
previous_channel_button.on_click(previous_channel)
next_channel_button.on_click(next_channel)

display(HBox([previous_button,next_button,previous_channel_button,next_channel_button]))
display(plot_output)
load_channel()
show_trial()

Output()

In [28]:
print(review_decisions)
print(review_file.resolve())

{(0, 0): 'Accepted'}
/Users/sreyashravinuthala/Downloads/Alec/Output/DA069_1_review_decisions.csv


In [29]:
def _preview_artifact_weight_sweep(artifact_matrix, channel_indices, applied_settings):
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display

    # Weights to test. Edit this range as needed.
    row_weights = np.round(np.arange(0, 4.01, 0.01), 2)

    def clean_matrix_at_weight(artifact_matrix, row_weight):
        A = np.asarray(artifact_matrix)

        if A.ndim != 2 or not np.isfinite(A).all() or np.any(A < 0):
            raise ValueError("Expected a finite, nonnegative channel x trial matrix.")

        if not np.isfinite(row_weight) or row_weight < 0:
            raise ValueError("Row weight must be finite and nonnegative.")

        channels = list(range(A.shape[0]))
        trials = list(range(A.shape[1]))

        while channels and trials:
            submatrix = A[np.ix_(channels, trials)]

            if not submatrix.any():
                break

            row_sums = np.sum(submatrix, axis=1) * row_weight
            col_sums = np.sum(submatrix, axis=0)

            max_row_idx = int(np.argmax(row_sums))
            max_col_idx = int(np.argmax(col_sums))

            if row_sums[max_row_idx] >= col_sums[max_col_idx]:
                channels.pop(max_row_idx)
            else:
                trials.pop(max_col_idx)

        return channels, trials

    # Local copies. Your original results are not modified.
    sweep_matrix = np.asarray(artifact_matrix).copy()
    sweep_channel_indices = np.asarray(channel_indices).copy()

    weight_sweep_results = {}
    sweep_records = []

    for weight in row_weights:
        kept_rows_at_weight, kept_trials_at_weight = clean_matrix_at_weight(sweep_matrix, weight)
        kept_channels_at_weight = sweep_channel_indices[kept_rows_at_weight].tolist()

        remaining = sweep_matrix[np.ix_(kept_rows_at_weight, kept_trials_at_weight)]
        remaining_flags = int(np.count_nonzero(remaining))

        weight_sweep_results[float(weight)] = {
            "channels": kept_channels_at_weight,
            "trials": kept_trials_at_weight,
            "matrix_rows": kept_rows_at_weight
        }

        sweep_records.append({
            "Row weight": float(weight),
            "Kept channels": len(kept_channels_at_weight),
            "Kept trials": len(kept_trials_at_weight),
            "Kept channel/trial pairs": len(kept_channels_at_weight) * len(kept_trials_at_weight),
            "Remaining flags": remaining_flags
        })

    weight_sweep_table = pd.DataFrame(sweep_records)

    # Your original channel and trial plots.
    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, constrained_layout=True)

    axes[0].plot(weight_sweep_table["Row weight"], weight_sweep_table["Kept channels"], marker="o", color="steelblue")
    axes[0].axhline(sweep_matrix.shape[0], color="gray", linestyle=":", label="Channels before cleaning")
    axes[0].set_ylabel("Kept channels")
    axes[0].legend()

    axes[1].plot(weight_sweep_table["Row weight"], weight_sweep_table["Kept trials"], marker="o", color="darkorange")
    axes[1].axhline(sweep_matrix.shape[1], color="gray", linestyle=":", label="Trials before cleaning")
    axes[1].set_ylabel("Kept trials")
    axes[1].set_xlabel("Row-sum weight")
    axes[1].legend()

    for ax in axes:
        ax.grid(alpha=0.2)
        ax.set_ylim(bottom=0)

    start, end = applied_settings["section_time_range"]
    bands = ", ".join(applied_settings["included_bands"])

    fig.suptitle(f"Row-weight sweep | {start:g} to {end:g} s | Bands: {bands}")

    weight_sweep_fig = fig
    
    plt.show()
    plt.close(fig)

    # Additional plot: retained channel/trial pairs and maximum.
    best_pairs = weight_sweep_table["Kept channel/trial pairs"].max()
    best_weights = weight_sweep_table.loc[weight_sweep_table["Kept channel/trial pairs"] == best_pairs]

    fig, ax = plt.subplots(figsize=(12, 5))

    ax.plot(weight_sweep_table["Row weight"], weight_sweep_table["Kept channel/trial pairs"], marker="o", color="seagreen")
    ax.scatter(best_weights["Row weight"], best_weights["Kept channel/trial pairs"], color="crimson", s=90, zorder=3, label=f"Maximum: {best_pairs:,} pairs")
    ax.set_xlabel("Row-sum weight")
    ax.set_ylabel("Kept channel/trial pairs")
    ax.set_title(f"Kept channel/trial pairs | {start:g} to {end:g} s | Bands: {bands}")
    ax.grid(alpha=0.2)
    ax.set_ylim(bottom=0)
    ax.legend()

    plt.tight_layout()
    plt.show()
    plt.close(fig)

    # Same table, with maximum-retention rows highlighted.
    def highlight_maximum(row):
        style = "background-color: #fff3bf; color: #111111; font-weight: bold;" if row["Kept channel/trial pairs"] == best_pairs else ""
        return [style] * len(row)

    display(weight_sweep_table.style.apply(highlight_maximum, axis=1).format({"Row weight": "{:.2f}"}))

    return weight_sweep_table, weight_sweep_fig, fig

# Run the preview and keep its table and figures for saving.
try:
    weight_sweep_table, weight_sweep_fig, kept_pairs_fig = (
        _preview_artifact_weight_sweep(
            section_artifacts,
            section_channel_indices,
            section_applied_settings,
        )
    )
finally:
    del _preview_artifact_weight_sweep

,Row weight,Kept channels,Kept trials,Kept channel/trial pairs,Remaining flags
0,0.00,132,34,4488,0
1,0.01,132,34,4488,0
2,0.02,132,34,4488,0
3,0.03,132,34,4488,0
4,0.04,132,34,4488,0
5,0.05,132,34,4488,0
6,0.06,132,34,4488,0
7,0.07,132,34,4488,0
8,0.08,132,34,4488,0
9,0.09,132,34,4488,0


In [30]:
#Settings
row_weight = 1.5

def clean_artifact_matrix(artifact_matrix):
    channels = set(range(artifact_matrix.shape[0]))
    trials = set(range(artifact_matrix.shape[1]))

    A = artifact_matrix.copy()

    while True:
        submatrix = A[np.ix_(list(channels), list(trials))]
        if not submatrix.any():
            break  # Done: No artifacts left

        # Count artifacts
        row_sums = np.sum(submatrix, axis=1)*row_weight
        col_sums = np.sum(submatrix, axis=0)

        max_row_idx = np.argmax(row_sums)
        max_col_idx = np.argmax(col_sums)

        if row_sums[max_row_idx] >= col_sums[max_col_idx]:
            # Drop channel
            channel_to_remove = list(channels)[max_row_idx]
            channels.remove(channel_to_remove)
        else:
            # Drop trial
            trial_to_remove = list(trials)[max_col_idx]
            trials.remove(trial_to_remove)

    return list(channels), list(trials)


# Use the FINAL selected-window matrix.
kept_rows, trial = clean_artifact_matrix(section_artifacts)

# Convert matrix rows back to original zero-based channel indices.
channel = np.asarray(section_channel_indices)[kept_rows].tolist()

# Proposed additional exclusions among the channels/trials evaluated here.
removed_channels = sorted(set(section_channel_indices.tolist()) - set(channel))
removed_trials = sorted(set(range(section_artifacts.shape[1])) - set(trial))

print("All indices below are zero-based.")
print(f"Channels kept ({len(channel)}): {channel}")
print(f"Trials kept ({len(trial)}): {trial}")
print(f"Additional channels to exclude (Beyond SoZ and by eye rejection) ({len(removed_channels)}): {removed_channels}")
print(f"Trials to exclude ({len(removed_trials)}): {removed_trials}")

All indices below are zero-based.
Channels kept (60): [4, 5, 6, 7, 8, 12, 13, 21, 26, 27, 40, 41, 42, 44, 45, 46, 47, 52, 54, 56, 57, 58, 60, 61, 62, 63, 64, 65, 66, 68, 81, 93, 96, 98, 99, 100, 101, 105, 110, 111, 112, 114, 116, 118, 119, 120, 121, 122, 123, 124, 135, 136, 138, 145, 153, 154, 156, 157, 159, 160]
Trials kept (112): [1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20, 22, 23, 25, 27, 28, 30, 31, 32, 33, 35, 36, 39, 41, 42, 43, 45, 46, 48, 49, 50, 55, 57, 58, 59, 60, 61, 62, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 79, 80, 81, 83, 84, 85, 86, 87, 88, 90, 91, 92, 93, 94, 95, 97, 98, 99, 100, 101, 102, 103, 105, 107, 109, 110, 112, 113, 114, 115, 116, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 134, 137, 138, 139, 140, 141, 142, 143]
Additional channels to exclude (Beyond SoZ and by eye rejection) (72): [9, 10, 11, 14, 20, 22, 23, 24, 25, 28, 34, 35, 36, 37, 38, 39, 43, 48, 50, 51, 53, 55, 59, 67, 69, 70, 71, 72, 73, 74,

In [31]:
# channel and trial are the KEEP lists returned by the cleaning cell.
kept_channels = np.asarray(channel, dtype=int)
kept_trials = np.asarray(trial, dtype=int)

# Map original channel numbers to rows in the selected-window matrix.
channel_to_row = {int(ch): row for row, ch in enumerate(section_channel_indices)}
kept_rows = np.array([channel_to_row[int(ch)] for ch in kept_channels], dtype=int)

kept_artifact_matrix = section_artifacts[np.ix_(kept_rows, kept_trials)]

if kept_artifact_matrix.shape[0] == 0 or kept_artifact_matrix.shape[1] == 0:
    print("No retained channel/trial matrix to plot.")

else:
    max_bands = len(section_applied_settings["included_bands"])

    fig, ax = plt.subplots(figsize=(16, 10))
    im = ax.imshow(kept_artifact_matrix, aspect="auto", interpolation="nearest", cmap="viridis", vmin=0, vmax=max_bands)

    fig.colorbar(im, ax=ax, label="Number of selected QC bands flagged", ticks=np.arange(max_bands + 1))

    x_ticks = np.arange(0, len(kept_trials), max(1, int(np.ceil(len(kept_trials) / 15))))
    y_ticks = np.arange(0, len(kept_channels), max(1, int(np.ceil(len(kept_channels) / 25))))

    ax.set_xticks(x_ticks)
    ax.set_xticklabels(kept_trials[x_ticks])
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(kept_channels[y_ticks])

    start, end = section_applied_settings["section_time_range"]
    bands = ", ".join(section_applied_settings["included_bands"])

    ax.set_xlabel("Original trial (zero-based)")
    ax.set_ylabel("Original channel (zero-based)")
    ax.set_title(f"Artifact Matrix — Kept Channels and Trials | {start:g} to {end:g} s\nBands: {bands}")

    plt.tight_layout()
    plt.show()

    print(f"Kept: {len(kept_channels)} channels × {len(kept_trials)} trials")
    print(f"Remaining flagged pairs: {np.count_nonzero(kept_artifact_matrix)}")

Kept: 60 channels × 112 trials
Remaining flagged pairs: 0


In [32]:
import numpy as np
import matplotlib.pyplot as plt
import mne
import ipywidgets as widgets
from IPython.display import display, clear_output


def build_kept_trial_viewer():
    required = ["epochs", "channel", "trial", "section_artifacts", "section_channel_indices", "section_applied_settings", "freqs", "n_cycles"]
    missing = [name for name in required if name not in globals()]

    if missing:
        raise RuntimeError("Run the section matrix and cleaning cells first. Missing: " + ", ".join(missing))

    ep = epochs
    kept_ch = np.sort(np.asarray(channel, dtype=int))
    kept_tr = np.sort(np.asarray(trial, dtype=int))
    window = np.asarray(section_applied_settings["section_time_range"], dtype=float).copy()
    frequencies = np.asarray(freqs).copy()
    cycles = np.asarray(n_cycles).copy()
    times = ep.times.copy()
    inside = (times >= window[0]) & (times <= window[1])

    if not len(kept_ch) or not len(kept_tr):
        raise ValueError("No kept channels/trials are available to view.")

    if np.any(kept_ch < 0) or np.any(kept_ch >= len(ep.ch_names)) or np.any(kept_tr < 0) or np.any(kept_tr >= len(ep)):
        raise ValueError("The keep lists do not match these epochs.")

    if inside.sum() < 2:
        raise ValueError("The selected window must contain at least two samples.")

    # Map original channels back to rows in the section matrix.
    channel_to_row = {int(ch): row for row, ch in enumerate(section_channel_indices)}

    if any(int(ch) not in channel_to_row for ch in kept_ch):
        raise ValueError("The section matrix has changed. Rerun the cleaning cell.")

    kept_rows = [channel_to_row[int(ch)] for ch in kept_ch]
    final_matrix = np.asarray(section_artifacts)[np.ix_(kept_rows, kept_tr)]

    if not np.isfinite(final_matrix).all() or np.any(final_matrix != 0):
        raise ValueError("These keep lists are not flag-free in the current section matrix. Rerun the cleaning cell.")

    # This viewer has its own state, separate from your first viewer.
    state = {"channel_position": 0, "trial_position": 0}
    cache = {}

    output = widgets.Output()
    progress = widgets.HTML()

    previous_trial = widgets.Button(description="Previous trial")
    next_trial = widgets.Button(description="Next trial")
    previous_channel = widgets.Button(description="Previous channel")
    next_channel = widgets.Button(description="Next channel")

    def load_channel(ch):
        if cache.get("channel") == ch:
            return

        cache.clear()
        print(f"Computing TFR for {ep.ch_names[ch]}...")

        raw = ep.get_data(picks=[int(ch)])

        # Calculate on FULL epochs to avoid introducing crop-edge effects.
        power = mne.time_frequency.tfr_array_morlet(raw, sfreq=ep.info["sfreq"], freqs=frequencies, n_cycles=cycles, output="power", use_fft=True, decim=1, n_jobs=1, verbose=False)
        db = 10 * np.log10(power[:, 0, :, :] + np.finfo(float).eps)
        del power

        # Match the first viewer's display normalization.
        # This uses all original trials/time for this channel.
        # It does NOT recalculate artifact-detection thresholds.
        center = np.median(db, axis=(0, 2))
        spread = np.std(db, axis=(0, 2))
        spread = np.where(spread > 0, spread, 1)

        # Store only kept trials and the selected time window for display.
        cropped = db[np.ix_(kept_tr, np.arange(len(frequencies)), np.flatnonzero(inside))]
        normalized = (cropped - center[None, :, None]) / spread[None, :, None]

        cache.update(channel=ch, tfr=normalized, raw=raw[kept_tr, 0, :][:, inside])

    def show():
        ch_pos = state["channel_position"]
        tr_pos = state["trial_position"]

        ch = int(kept_ch[ch_pos])
        tr = int(kept_tr[tr_pos])

        progress.value = f"Kept channel {ch_pos + 1}/{len(kept_ch)} | Kept trial {tr_pos + 1}/{len(kept_tr)} | Matrix indices: channel {ch}, trial {tr}"

        previous_trial.disabled = tr_pos == 0
        next_trial.disabled = tr_pos == len(kept_tr) - 1
        previous_channel.disabled = ch_pos == 0
        next_channel.disabled = ch_pos == len(kept_ch) - 1

        with output:
            clear_output(wait=True)
            load_channel(ch)

            fig, (ax, ax_raw) = plt.subplots(2, 1, figsize=(13, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1]}, constrained_layout=True)

            im = ax.pcolormesh(times[inside], frequencies, cache["tfr"][tr_pos], shading="auto", cmap="viridis", vmin=-5, vmax=5)

            ax.set_yscale("log")

            ticks = np.array([2, 4, 10, 30, 70, 100, 200, 250])
            ticks = ticks[(ticks >= frequencies[0]) & (ticks <= frequencies[-1])]

            ax.set_yticks(ticks)
            ax.set_yticklabels([f"{value:g}" for value in ticks])
            ax.set_ylim(frequencies[0], frequencies[-1])
            ax.set_ylabel("Frequency (Hz)")
            ax.set_title(f"Final QC — {ep.ch_names[ch]} — Trial {tr + 1} | {window[0]:g} to {window[1]:g} s")

            ax_raw.plot(times[inside], cache["raw"][tr_pos], color="black", linewidth=0.7)
            ax_raw.set_ylabel("Stored amplitude")
            ax_raw.set_xlabel("Time relative to decision (s)")

            for axis in (ax, ax_raw):
                axis.set_xlim(window[0], window[1])

                if window[0] <= 0 <= window[1]:
                    axis.axvline(0, color="black", linestyle="--", linewidth=0.8)

            fig.colorbar(im, ax=[ax, ax_raw], label="Normalized log power (SD from channel median)")
            plt.show()
            plt.close(fig)

    def move(which, step):
        limit = len(kept_ch) if which == "channel_position" else len(kept_tr)
        state[which] = max(0, min(limit - 1, state[which] + step))
        show()

    previous_trial.on_click(lambda _: move("trial_position", -1))
    next_trial.on_click(lambda _: move("trial_position", 1))
    previous_channel.on_click(lambda _: move("channel_position", -1))
    next_channel.on_click(lambda _: move("channel_position", 1))

    display(widgets.VBox([progress, widgets.HBox([previous_trial, next_trial, previous_channel, next_channel]), output]))

    show()

    return {"show": show, "state": state, "channels": kept_ch, "trials": kept_tr}


kept_trial_viewer = build_kept_trial_viewer()

In [33]:
#Saves (Save to an output folder - shouldnt be very complex)
#PNGs of original full epoch artifact matrix. 4 images (1 is Low, Med, and High together; 2,3, and 4 are the low, mid, and high artifact matrix images 
#Matrix CSV saves for each of the full epoch artifact matrixes 

#Do the same for selected window artifact matrix for the images and CSV

#Save row weight sweep and kept channel/trial pair images
#Save the weight_sweep_table

#final trial and channel list
    #Save row weight 
    #Save SoZ rejected
    #Save by eye rejection 
    #Channels and trials rejected 
    #Channels and trials kept 

In [34]:
# Save artifact matrices, sweep results, and final rejection/keep lists.
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Change this folder if desired. Rerunning overwrites matching filenames.
output_dir = file_path.parent / "Output" / file_path.stem
output_dir.mkdir(parents=True, exist_ok=True)


# ---------- Matrix PNG + CSV helper ----------
def save_artifact_matrix(matrix, filename, channel_ids, title, vmax):
    matrix = np.asarray(matrix)
    channel_ids = np.asarray(channel_ids, dtype=int)

    # CSV rows retain original channel indices; columns are trial indices.
    pd.DataFrame(
        matrix,
        index=pd.Index(channel_ids, name="channel"),
        columns=np.arange(matrix.shape[1]),
    ).to_csv(output_dir / f"{filename}.csv")

    fig, ax = plt.subplots(figsize=(16, 10))

    if matrix.size:
        im = ax.imshow(
            matrix, aspect="auto", interpolation="nearest",
            cmap="viridis", vmin=0, vmax=vmax,
        )
        fig.colorbar(
            im, ax=ax, ticks=np.arange(vmax + 1),
            label="Artifact flag" if vmax == 1 else "Number of QC bands flagged",
        )
        ticks = np.arange(0, len(channel_ids), 5)
        ax.set_yticks(ticks)
        ax.set_yticklabels(channel_ids[ticks])
    else:
        ax.text(
            0.5, 0.5, "No channels/trials available",
            ha="center", va="center", transform=ax.transAxes,
        )

    ax.set_xlabel("Trial (zero-based)")
    ax.set_ylabel("Original channel (zero-based)")
    ax.set_title(title)
    fig.savefig(output_dir / f"{filename}.png", dpi=300, bbox_inches="tight")
    plt.close(fig)


# ---------- Original full detection matrices: 4 PNGs + 4 CSVs ----------
export_full_matrices = {
    "Combined": artifacts,
    "Low": artifacts_low,
    "Mid": artifacts_mid,
    "High": artifacts_high,
}

for export_band, export_matrix in export_full_matrices.items():
    save_artifact_matrix(
        export_matrix,
        f"full_epoch_{export_band.lower()}",
        np.arange(artifacts.shape[0]),
        f"Full detection window | {export_band} | "
        f"{artifact_detection_settings['analysis_time_range']} s",
        vmax=3 if export_band == "Combined" else 1,
    )


# ---------- Selected window matrices: 4 PNGs + 4 CSVs ----------
export_section_matrices = {
    "Combined": section_artifacts,
    **section_results["band_matrices"],
}

export_start, export_end = section_applied_settings["section_time_range"]
export_bands = section_applied_settings["included_bands"]

for export_band, export_matrix in export_section_matrices.items():
    export_detail = (
        f"Bands: {', '.join(export_bands)}"
        if export_band == "Combined"
        else f"Buffer: {section_applied_settings['buffer_ms'][export_band]:g} ms"
    )

    save_artifact_matrix(
        export_matrix,
        f"selected_window_{export_band.lower()}",
        section_channel_indices,
        f"Selected window | {export_start:g} to {export_end:g} s | "
        f"{export_band}\n{export_detail}",
        vmax=len(export_bands) if export_band == "Combined" else 1,
    )


# ---------- Sweep plots and table ----------
weight_sweep_fig.savefig(
    output_dir / "row_weight_sweep.png", dpi=300, bbox_inches="tight"
)
kept_pairs_fig.savefig(
    output_dir / "kept_channel_trial_pairs.png", dpi=300, bbox_inches="tight"
)
weight_sweep_table.to_csv(output_dir / "weight_sweep_table.csv", index=False)


# ---------- Final lists: all indices are original, zero-based ----------
# `channel` and `trial` are the KEEP lists from your final cleaning cell.
export_lists = {
    "channels_kept": channel,
    "trials_kept": trial,
    "channels_rejected_soz": seizure_onset_channels,
    "channels_rejected_by_eye": rejected_channels,
    "channels_rejected_by_cleaning": (
        set(section_channel_indices) - set(channel)
    ),
    "channels_rejected_all": (
        set(range(artifacts.shape[0])) - set(channel)
    ),
    "trials_rejected": (
        set(range(artifacts.shape[1])) - set(trial)
    ),
}

export_lists = {
    name: sorted(set(map(int, values)))
    for name, values in export_lists.items()
}

for export_name, export_values in export_lists.items():
    export_column = "channel" if export_name.startswith("channels") else "trial"
    pd.DataFrame({export_column: export_values}).to_csv(
        output_dir / f"{export_name}.csv", index=False
    )


# Save chosen row weight, window settings, and all final lists together.
export_summary = {
    "source_file": str(file_path),
    "index_base": 0,
    "row_weight": float(row_weight),
    "detection_time_range_s": list(map(
        float, artifact_detection_settings["analysis_time_range"]
    )),
    "selected_time_range_s": [float(export_start), float(export_end)],
    "buffer_ms": {
        band: float(value)
        for band, value in section_applied_settings["buffer_ms"].items()
    },
    "included_bands": list(export_bands),
    **export_lists,
}

with (output_dir / "final_selection.json").open("w", encoding="utf-8") as f:
    json.dump(export_summary, f, indent=2)

print(f"Saved all outputs to:\n{output_dir}")

Saved all outputs to:
/Users/sreyashravinuthala/Downloads/Alec/Output/DA069_1_preprocessed_epoched


In [35]:
output_dir = file_path.parent / "Output" / file_path.stem
output_dir.mkdir(parents=True, exist_ok=True)
print(output_dir)

/Users/sreyashravinuthala/Downloads/Alec/Output/DA069_1_preprocessed_epoched


In [36]:
#Hald Length morlet proof 


In [37]:
bndf.close()
print("Closed")

Closed
